# Geração de Dados - Camada Bronze

Este notebook gera dados fictícios para simular um ambiente de sistema bancário com suas respectivas pendências:
* **Agências**: 40 agências fixas.
* **Clientes**: 500 clientes com histórico de 12 meses.
* **Contratos**: 800 contratos com histórico de 12 meses.

In [0]:
%python
# Instalar biblioteca para geração de dados fictícios
%pip install faker

In [0]:
%python
# Importação de bibliotecas necessárias
from pyspark.sql import SparkSession
import random
from faker import Faker
from datetime import datetime, timedelta

# Inicializar SparkSession e Faker com locale brasileiro
spark = SparkSession.builder.getOrCreate()
fake = Faker("pt_BR")

print("Configuração concluída")

## 1. Tabela: Agências

Gera 40 agências bancárias fixas com:
* ID único (1-40)
* Nome, cidade e UF gerados com Faker

In [0]:
%python
# Gerar dados de 40 agências
agencias = [
    (i, f"Agência {i}", fake.city(), fake.state_abbr()) 
    for i in range(1, 41)
]

# Criar DataFrame
df_agencias = spark.createDataFrame(
    agencias, 
    ["agencia_id", "nome_agencia", "cidade", "uf"]
)

# Salvar como tabela Delta
df_agencias.write.mode("overwrite").saveAsTable("bronze_agencias")

print(f"✓ Tabela bronze_agencias criada com {df_agencias.count()} registros")
display(df_agencias)

## 2. Tabela: Clientes

Gera 500 clientes únicos particionados por 12 meses:
* Dados brasileiros (nome, CPF, email, telefone)
* 10% emails nulos e 5% telefones nulos
* Vinculados a agências aleatórias
* Data de cadastro
* Particionados por `data_particao` (12 meses de histórico)

In [0]:
%python
# Gerar 500 clientes únicos
clientes = []
for i in range(1, 501):
    agencia_id = random.randint(1, 40)
    
    clientes.append((
        i,
        fake.name(),
        fake.cpf(),
        fake.email() if random.random() > 0.1 else None,  # 10% nulos
        fake.phone_number() if random.random() > 0.05 else None,  # 5% nulos
        agencia_id,
        str(fake.date_between(start_date="-2y", end_date="-6m"))  # Data de cadastro
    ))

# Particionar para 12 meses (mesmos clientes em cada mês)
clientes_particionado = []
for mes in range(12):
    data_particao = (datetime.today() - timedelta(days=30 * mes)).strftime("%Y-%m-%d")
    for c in clientes:
        clientes_particionado.append(c + (data_particao,))

# Criar DataFrame
df_clientes = spark.createDataFrame(
    clientes_particionado,
    ["cliente_id", "nome", "cpf_cnpj", "email", "telefone",
     "agencia_id", "data_cadastro", "data_particao"]
)

# Salvar como tabela Delta particionada
df_clientes.write \
    .mode("overwrite") \
    .partitionBy("data_particao") \
    .saveAsTable("bronze_clientes")

print(f"Tabela bronze_clientes criada com {df_clientes.count()} registros")
print(f"500 clientes únicos × 12 meses")
display(df_clientes.filter(f"data_particao = '{(datetime.today()).strftime('%Y-%m-%d')}'").limit(10))

## 3. Tabela: Contratos

Gera 800 contratos particionados por 12 meses:
* Vinculados a clientes e suas respectivas agências
* Tipos: Seguro, Financiamento, Empréstimo Pessoal, Consignado, Cartão de Crédito
* Dados de contrato: datas (contrato, início, fim), valor total, valor da parcela
* Quantidade de parcelas e parcelas em atraso
* **Garantia vinculada**: Financiamentos têm 80% com garantia (Imóvel, Veículo, Aplicações); 20% nulos
* Particionados por `data_particao` (12 meses de histórico)

In [0]:
%python
# Tipos de contratos do mercado financeiro
tipos_contrato = [
    "Seguro", 
    "Financiamento", 
    "Empréstimo Pessoal", 
    "Consignado", 
    "Cartão de Crédito"
]

# Tipos de garantia para contratos
tipos_garantia = ["Imóvel", "Veículo", "Aplicações Financeiras"]

# Gerar 800 contratos
contratos = []
for i in range(1, 801):
    cliente_id = random.randint(1, 500)
    agencia_id = clientes[cliente_id - 1][5]  # Mesma agência do cliente (índice 5)
    
    # Gerar dados do contrato
    tipo_contrato = random.choice(tipos_contrato)
    valor_total = round(random.uniform(5000, 50000), 2)
    qtd_parcelas = random.randint(6, 60)
    valor_parcela = round(valor_total / qtd_parcelas, 2)
    parcelas_em_atraso = random.randint(0, min(5, qtd_parcelas))  # Até 5 parcelas atrasadas
    
    # Garantia vinculada (obrigatória para Financiamento)
    if tipo_contrato == "Financiamento":
        garantia = random.choice(tipos_garantia) if random.random() > 0.2 else None  # 80% com garantia
    else:
        garantia = random.choice(tipos_garantia) if random.random() > 0.8 else None  # 20% com garantia
    
    # Datas do contrato
    data_contrato = fake.date_between(start_date="-2y", end_date="-3m")
    data_inicio = data_contrato + timedelta(days=random.randint(1, 15))
    data_fim = data_inicio + timedelta(days=qtd_parcelas * 30)  # Aproximadamente mensal
    
    contratos.append((
        i,
        cliente_id,
        agencia_id,
        tipo_contrato,
        garantia,
        str(data_contrato),
        str(data_inicio),
        str(data_fim),
        valor_total,
        valor_parcela,
        qtd_parcelas,
        parcelas_em_atraso
    ))

# Particionar para 12 meses
contratos_particionado = []
for mes in range(12):
    data_particao = (datetime.today() - timedelta(days=30 * mes)).strftime("%Y-%m-%d")
    for ct in contratos:
        contratos_particionado.append(ct + (data_particao,))

# Criar DataFrame
df_contratos = spark.createDataFrame(
    contratos_particionado,
    ["contrato_id", "cliente_id", "agencia_id", "tipo_contrato", "garantia",
     "data_contrato", "data_inicio", "data_fim", "valor_total",
     "valor_parcela", "quantidade_parcelas", "parcelas_em_atraso", "data_particao"]
)

# Salvar como tabela Delta particionada
df_contratos.write \
    .mode("overwrite") \
    .partitionBy("data_particao") \
    .saveAsTable("bronze_contratos")

print(f"Tabela bronze_contratos criada com {df_contratos.count()} registros")
print(f"800 contratos únicos × 12 meses")
print(f"Tipos: {', '.join(tipos_contrato)}")
display(df_contratos.filter(f"data_particao = '{(datetime.today()).strftime('%Y-%m-%d')}'").limit(10))